# 02 — NYC Taxi Bronze to Silver
## 1. Read Bronze Data

In [ ]:
df_taxi = spark.read.parquet(
    "Files/bronze/nyc_taxi/yellow_tripdata_2025-06.parquet",
    "Files/bronze/nyc_taxi/yellow_tripdata_2025-07.parquet"
)

df_taxi.printSchema()

In [ ]:
print("Total rows:", df_taxi.count())

df_taxi.show(5, truncate=False)

## 2. Initial Data Quality Checks

In [ ]:
passenger_count = df_taxi.filter(df_taxi.passenger_count == 0).count()
print("Passenger count with 0: ", passenger_count)

In [ ]:
passenger_count_null = df_taxi.filter(df_taxi.passenger_count.isNull()).count()
print("Passenger Count Null: ", passenger_count_null)

In [ ]:
zero_duration_count = df_taxi.filter(df_taxi.tpep_pickup_datetime == df_taxi.tpep_dropoff_datetime).count()
print("Passenger with Zero Duration: ", zero_duration_count)

In [ ]:
zero_trip_distance = df_taxi.filter(df_taxi.trip_distance <= 0).count()
print("Trip with zero or below distance: ", zero_trip_distance)

In [ ]:
negative_fare_count = df_taxi.filter(df_taxi.fare_amount < 0).count()
print("Negative Fare Count: ", negative_fare_count)

## 3. Silver Data Transformations
### 3.1 Data Quality Flags

In [ ]:
df_silver = df_taxi.withColumn(
    "is_negative_fare",
    df_taxi.fare_amount < 0
)

In [ ]:
df_silver = df_silver.withColumn(
    "is_invalid_distance", df_silver.trip_distance <= 0
)

In [ ]:
df_silver = df_silver.withColumn(
    "is_zero_duration", df_silver.tpep_pickup_datetime == df_silver.tpep_dropoff_datetime
)

In [ ]:
df_silver = df_silver.withColumn(
    "is_null_passenger", df_silver.passenger_count.isNull()
)

df_silver = df_silver.withColumn(
    "is_zero_passenger", df_silver.passenger_count.eqNullSafe(0)
)

## 4. Write Initial Silver Table

Save the initial Silver dataset with data quality flags.

In [ ]:
df_check = df_silver

In [ ]:
print("Total rows:", df_check.count())
print("Total columns:", len(df_check.columns))

In [ ]:
df_check.select(
    "is_negative_fare",
    "is_invalid_distance",
    "is_zero_duration",
    "is_null_passenger",
    "is_zero_passenger"
).show(5)

## 5. Trip Date and Duration Transformations

Add trip date, calculate trip duration, and identify invalid durations.

In [ ]:
df_taxi_silver = df_silver

print(df_taxi_silver.columns)

In [ ]:
from pyspark.sql.functions import to_date, substring

df_taxi_silver = df_taxi_silver.withColumn(
    "trip_date",
    to_date(substring("tpep_pickup_datetime", 1, 10))
)

In [ ]:
from pyspark.sql.functions import unix_timestamp

df_taxi_silver = df_taxi_silver.withColumn(
    "trip_duration_minutes",
    (
    unix_timestamp("tpep_dropoff_datetime")
    - unix_timestamp("tpep_pickup_datetime")
    ) / 60.0
)

In [ ]:
negative_duration_count = df_taxi_silver.filter(
    df_taxi_silver.trip_duration_minutes < 0
    ).count()

print("Negative Duration Count: ", negative_duration_count)

In [ ]:
long_duration_count = df_taxi_silver.filter(
    df_taxi_silver.trip_duration_minutes > 1440
    ).count()

print("Long trip duration: ", long_duration_count)

In [ ]:
df_taxi_silver = df_taxi_silver.withColumn(
    "is_negative_duration",
    df_taxi_silver.trip_duration_minutes < 0
)

df_taxi_silver = df_taxi_silver.withColumn(
    "is_long_duration",
    df_taxi_silver.trip_duration_minutes > 1440
)

In [ ]:
df_taxi_silver.groupBy(
    "is_negative_duration",
    "is_long_duration"
).count().show()

## 6. Write Enriched Silver Table

Save the enriched taxi dataset with trip dates, durations, and quality flags.

In [ ]:
df_taxi_v2 = df_taxi_silver

## 7. Silver Data Validation

Check duplicate records, missing timestamps, and reporting date boundaries.

In [ ]:
df_check = df_taxi_v2

print("Rows:", df_check.count())
print("Columns:", len(df_check.columns))

In [ ]:
total_rows = df_taxi_v2.count()

unique_rows = df_taxi_v2.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", duplicate_rows)

In [ ]:
null_timestamps = df_taxi_v2.filter(
    (df_taxi_v2.tpep_pickup_datetime.isNull())
    |
    (df_taxi_v2.tpep_dropoff_datetime.isNull())
).count()

print("NULL Timestamps:", null_timestamps)

In [ ]:
out_of_range_count = df_taxi_v2.filter(
    (df_taxi_v2.trip_date < "2025-06-01")
    |
    (df_taxi_v2.trip_date >= "2025-08-01")
).count()

print("Out of range:", out_of_range_count)

## 8. Reporting Period and Deduplication

Flag trips within June–July 2025 and remove exact duplicate records.

In [ ]:
df_taxi_final = df_taxi_v2.withColumn(
    "is_in_reporting_period",
    (df_taxi_v2.trip_date >= "2025-06-01")
    &
    (df_taxi_v2.trip_date < "2025-08-01")
)

In [ ]:
df_taxi_final.groupBy("is_in_reporting_period").count().show()

In [ ]:
df_taxi_final = df_taxi_final.dropDuplicates()

print("Final Rows:", df_taxi_final.count())


## 9. Write Final Reporting Silver Table

Save the reporting-period flag and deduplicated records to silver_taxi_trips.

In [ ]:
(
    df_taxi_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_taxi_trips")
)

## 10. Final Output Validation

In [ ]:
df_check = spark.table("silver_taxi_trips")

print("Final Rows:", df_check.count())
print("Final Columns:", len(df_check.columns))